# 10. Batch processing and reporting

Running many cases and collecting the numbers. The tutorial scripts in `../tutorials/` are the production entry points; this notebook shows what they are made of and how to gather their outputs into one table.

## One case from a parameters file

`tutorials/arterial_processing_full_pipeline.py` reads a JSON like the one below, builds the processor arguments and runs the stages that are not skipped. Skipping stages you already have is how a cohort is re-run cheaply after a code change.

In [1]:
import json, os
from arterial_nb import *
with open(os.path.join(REPO_ROOT, "tutorials", "arterial_processing_params.json")) as handle:
    params = json.load(handle)
table([[k, v] for k, v in params.items()], columns=["parameter", "value"])

,parameter,value
0,case_dir,/path/to/case_dir
1,cta_nifti_path,
2,mode,extracranial_vessels
3,fast_segmentation,False
4,sampling_distance_mm,2.0
5,skip_segmentation,True
6,skip_centerline_extraction,True
7,skip_branching,False
8,skip_clipping,True
9,skip_vessel_labelling,True


In [2]:
import sys; sys.path.insert(0, os.path.join(REPO_ROOT, "tutorials"))
from arterial_processing_full_pipeline import load_args_from_params_json
from arterial.run.processor import ArterialProcessor

params.update(case_dir=case("batch_demo"), cta_nifti_path=fixture("cta.nii.gz"),
              skip_segmentation=True, skip_centerline_extraction=True, skip_landmark_detection=True, skip_vessel_labelling=True, skip_feature_extraction=True, skip_access_prediction=True)
with quiet():
    times = ArterialProcessor(load_args_from_params_json(params)).perform_analysis()
table([[k.replace("_time", ""), round(v, 2)] for k, v in times.items()], columns=["stage", "s"]).T


Arterial framework for automated characterization of vascular tortuosity.
Copyright 2022-2026 Vall d'Hebron Research Institute (VHIR) and Universitat de Barcelona (UB), Barcelona, Spain.
Code licensed under the PolyForm Noncommercial License 1.0.0. Noncommercial use only.



,0,1,2,3,4,5,6
stage,segmentation,centerline_extraction,landmark_detection,vessel_labelling,feature_extraction,access_prediction,total
s,0.0,0.0,0.0,0.0,0.0,0.0,0.0


The timing dictionary is returned by every run; the batch script writes it to the case log, which is the easiest way to plan compute for a cohort.

## A cohort loop

`tutorials/batch_arterial_processing.py` is a template: it reads identifiers from a spreadsheet, prepares one case directory per patient (symlinking the NIfTI), writes the parameters file, calls the single-case entry point and logs failures without stopping. The loop itself is a few lines:

In [3]:
import inspect, arterial_processing_full_pipeline
print(inspect.getsource(arterial_processing_full_pipeline.main)[:1200])

def main(parameters_file_path):
    with open(parameters_file_path, 'r') as f:
        params = json.load(f)

    case_dir = params.get('case_dir', None)
    assert case_dir is not None and case_dir != "", "case_dir is required"
    cta_nifti_path = params.get('cta_nifti_path', os.path.join(case_dir, "cta.nii.gz"))
    assert cta_nifti_path is not None and cta_nifti_path != "", "cta_nifti_path is required"

    os.makedirs(case_dir, exist_ok=True)
    
    # Setup case-specific logging
    log_filename = f"arterial_processing_{datetime.now().strftime('%Y%m%d_%H%M%S')}.log"
    log_filepath = os.path.join(case_dir, log_filename)
    
    logger = logging.getLogger(case_dir)
    logger.setLevel(logging.DEBUG)
    
    # Clear any existing handlers to avoid duplication
    logger.handlers = []
    
    # File handler
    file_handler = logging.FileHandler(log_filepath)
    file_handler.setLevel(logging.DEBUG)
    
    # Console handler
    console_handler = logging.StreamHandler()
    con

## Collecting results

Each case leaves its measurements in a handful of files. Reading them into one row per case gives the table most studies start from.

In [4]:
from arterial.io.load_and_save_operations import load_pickle, load_json

def summarise_case(mode_dir, case_id):
    graph = load_pickle(os.path.join(mode_dir, "local_graph.pickle"))
    row = {"case": case_id, "aortic_arch_type": graph.graph["aortic_arch_type"], "bovine_arch": graph.graph["bovine_arch"], "arsa": graph.graph["arsa"]}
    for vessel, feats in graph.graph["segment_features"].items():
        for name in ["length", "mean_diameter", "tortuosity_index"]:
            if name in feats:
                row[f"{vessel} {name}"] = round(float(feats[name]), 2)
    access_dir = os.path.join(mode_dir, "access_prediction")
    for side in ["left", "right"]:
        path = os.path.join(access_dir, f"femoral_{side}", "access_prediction.json")
        if os.path.isfile(path):
            row[f"femoral {side} P(difficult)"] = round(float(load_json(path)["mean"]), 3)
    return row

cohort = table([summarise_case(derived(), "fixture")])
cohort.T

,0
case,fixture
aortic_arch_type,1
bovine_arch,0
arsa,0
AA length,105.63
AA mean_diameter,25.54
AA tortuosity_index,0.15
BT length,70.06
BT mean_diameter,13.09
BT tortuosity_index,0.44


In [5]:
cohort.to_csv(case("cohort_summary.csv"), index=False)
print("written", case("cohort_summary.csv"), "with", cohort.shape[1], "columns")

written /home/perecanals/arterial_open_source_test/arterial/notebooks/case/cohort_summary.csv with 46 columns


## Notes

- Keep the raw products; the summary table can always be rebuilt from them, the reverse is not true.
- Run the fast tier of the tests (`ARTERIAL_SKIP_SLOW=1 python -m unittest discover -s tests`) after updating the code and before re-running a cohort.
- The access predictions exist only for the femoral anterior pathways; other columns stay empty.